YOLOv8 Training Helpers

Each function from `train.py` is placed in its own Python cell below. Run the cells in order: first the imports/config cell, then the function-definition cells, then any call cells you want to execute.

In [ ]:
# Imports and configuration
import os
import sys
import yaml
import zipfile
import shutil
from pathlib import Path
try:
    from ultralytics import YOLO
except Exception:
    YOLO = None

# Configuration (adjust paths as needed)
DATASET_DIR = os.path.abspath("C:\\Users\\bss10\\Desktop\\drishti\\drishti_kavach\\archive\\IDDDetectionsYOLODataset")
KAGGLE_DATASET = "redzapdos123/indian-driving-dataset-detections-yolov11"
EPOCHS = 10
BATCH_SIZE = 8
IMG_SIZE = 640
DEVICE = "cpu"
MODEL = "yolov8n.pt"

In [ ]:
# dataset_exists_and_valid
def dataset_exists_and_valid(dataset_path):
    required_dirs = ["train", "val", "test"]
    required_files = ["data.yaml"]
    if not os.path.isdir(dataset_path):
        return False
    for subdir in required_dirs:
        subdir_path = os.path.join(dataset_path, subdir)
        if not os.path.isdir(subdir_path):
            return False
    for filename in required_files:
        file_path = os.path.join(dataset_path, filename)
        if not os.path.isfile(file_path):
            return False
    return True

In [ ]:
# Call dataset_exists_and_valid
print('dataset_exists_and_valid ->', dataset_exists_and_valid(DATASET_DIR))

In [ ]:
# download_dataset_from_kaggle
def download_dataset_from_kaggle(dataset_id, target_dir):
    try:
        import kaggle
    except ImportError:
        raise RuntimeError("kaggle library not found. Install with: pip install kaggle")
    print(f"Downloading dataset from Kaggle: {dataset_id} -> {target_dir}")
    os.makedirs(target_dir, exist_ok=True)
    try:
        kaggle.api.dataset_download_files(dataset_id, path=target_dir, unzip=False)
        print("Dataset download completed.")
    except Exception as e:
        raise RuntimeError(f"Failed to download dataset: {e}")

In [ ]:
# Call download_dataset_from_kaggle
# Warning: running this cell will attempt to download the dataset via Kaggle API.
# Ensure kaggle.json is configured before running.
# download_dataset_from_kaggle(KAGGLE_DATASET, DATASET_DIR)
print('Download cell ready. Uncomment the call to run it.')

In [ ]:
# extract_dataset
def extract_dataset(target_dir):
    zip_files = [f for f in os.listdir(target_dir) if f.endswith('.zip')]
    if not zip_files:
        print("No ZIP files found to extract.")
        return
    for zip_file in zip_files:
        zip_path = os.path.join(target_dir, zip_file)
        print(f"Extracting: {zip_file}")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
        os.remove(zip_path)
        print(f"Extracted and removed: {zip_file}")

In [ ]:
# Call extract_dataset
# extract_dataset(DATASET_DIR)
print('Extract cell ready. Uncomment the call to run it.')

In [ ]:
# ensure_dataset_available
def ensure_dataset_available(dataset_dir, kaggle_dataset_id):
    if dataset_exists_and_valid(dataset_dir):
        print(f"Dataset found and validated at: {dataset_dir}")
        return
    print(f"Dataset not found or invalid at: {dataset_dir}. Attempting download...")
    download_dataset_from_kaggle(kaggle_dataset_id, dataset_dir)
    extract_dataset(dataset_dir)
    if not dataset_exists_and_valid(dataset_dir):
        raise RuntimeError("Dataset validation failed after download.")
    print(f"Dataset is now ready at: {dataset_dir}")

In [ ]:
# Call ensure_dataset_available
# Warning: this may download/extract data. Uncomment to run.
# ensure_dataset_available(DATASET_DIR, KAGGLE_DATASET)
print('Ensure-dataset cell ready. Uncomment the call to run it.')

In [ ]:
# validate_dataset_structure
def validate_dataset_structure(dataset_root):
    dataset_path = Path(dataset_root)
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset root directory not found: {dataset_root}")
    required_dirs = [
        "train/images",
        "train/labels",
        "val/images",
        "val/labels",
        "test/images",
        "test/labels",
    ]
    for dir_name in required_dirs:
        dir_path = dataset_path / dir_name
        if not dir_path.exists():
            raise FileNotFoundError(f"Required directory not found: {dir_path}")
        if not list(dir_path.glob("*")):
            print(f"Warning: {dir_path} is empty")
    data_yaml = dataset_path / "data.yaml"
    if not data_yaml.exists():
        raise FileNotFoundError(f"data.yaml not found: {data_yaml}")
    print(f"Dataset structure validated at: {dataset_path}")

In [ ]:
# Call validate_dataset_structure
# validate_dataset_structure(DATASET_DIR)
print('Validate-structure cell ready. Uncomment the call to run it.')

In [ ]:
# update_data_yaml
def update_data_yaml(dataset_root):
    dataset_path = Path(dataset_root)
    data_yaml_path = dataset_path / "data.yaml"
    with open(data_yaml_path, "r") as f:
        data = yaml.safe_load(f)
    data["path"] = str(dataset_path.absolute())
    data["train"] = str((dataset_path / "train/images").absolute())
    data["val"] = str((dataset_path / "val/images").absolute())
    data["test"] = str((dataset_path / "test/images").absolute())
    with open(data_yaml_path, "w") as f:
        yaml.dump(data, f, default_flow_style=False)
    print("data.yaml updated with absolute paths")
    return str(data_yaml_path)

In [ ]:
# Call update_data_yaml
# data_yaml_path = update_data_yaml(DATASET_DIR)
# print(data_yaml_path)
print('Update-data-yaml cell ready. Uncomment the call to run it.')

In [ ]:
# train_model
def train_model(data_yaml_path):
    if YOLO is None:
        raise RuntimeError('ultralytics.YOLO is not available. Install ultralytics to train.')
    print('Starting training with model:', MODEL)
    model = YOLO(MODEL)
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        device=DEVICE,
        patience=5,
        save=True,
        verbose=True,
        project="runs/detect",
        name="license_plate_detection",
    )
    return results

In [ ]:
# Call train_model
# Warning: running this will start model training and may take a long time.
# results = train_model(<path-to-updated-data.yaml>)
print('Train cell ready. Provide the path to data.yaml and uncomment the call to run it.')

In [ ]:
# print_training_summary
def print_training_summary(results=None):
    run_dir = Path("runs/detect/license_plate_detection")
    best_weights = run_dir / "weights/best.pt"
    last_weights = run_dir / "weights/last.pt"
    print("Training Complete!")
    if best_weights.exists():
        print('Best weights saved at:', best_weights.absolute())
    else:
        print('Best weights file not found.')
    if last_weights.exists():
        print('Last checkpoint saved at:', last_weights.absolute())
    print('All training artifacts saved at:', run_dir.absolute())

In [ ]:
# Call print_training_summary
# print_training_summary()
print('Summary cell ready. Uncomment the call to run it.')